# LangGraph Swarm Agents

## 학습 목표
- LangGraph + Swarm의 개념과 장점 이해
- 멀티 에이전트 패턴 비교 (Tool-calling, Supervisor, Swarm)
- Swarm 핵심 API 학습 및 실습
- 에이전트 간 핸드오프(handoff) 구현


## LangGraph + Swarm이란?
- [LangGraph Swarm 공식 문서](https://reference.langchain.com/python/langgraph/swarm/)


### Swarm의 개념
**Swarm**은 여러 에이전트가 협업하여 복잡한 작업을 수행하는 멀티 에이전트 시스템입니다.

### 핵심 특징
1. **경량화된 멀티 에이전트 오케스트레이션**: OpenAI의 Swarm 패턴에서 영감을 받음
2. **동적 핸드오프**: 에이전트 간 제어권을 동적으로 전환
3. **컨텍스트 유지**: 대화 상태와 히스토리를 유지하면서 에이전트 전환
4. **도구 기반 전환**: 특수한 핸드오프 도구를 통해 에이전트 간 이동


## 단일 에이전트 vs 멀티 에이전트


### 단일 에이전트 (Single Agent)
```
사용자 → Agent → 도구들 → 응답
```
**장점:**
- 간단한 구조
- 빠른 응답
- 관리 용이

**단점:**
- 복잡한 작업 처리 어려움
- 전문화된 작업 처리 제한
- 컨텍스트 관리 복잡


### 멀티 에이전트 (Multi-Agent)
```
사용자 → 조정자 → Agent1 (전문가1)
              ↓
              → Agent2 (전문가2)
              ↓
              → Agent3 (전문가3)
```
**장점:**
- 각 에이전트가 특정 도메인 전문화
- 복잡한 워크플로우 처리
- 병렬 처리 가능

**단점:**
- 복잡한 오케스트레이션 필요
- 디버깅 어려움
- 오버헤드 증가


## 멀티 에이전트 패턴 비교

### Tool-Calling 패턴
- **구조**: 단일 에이전트가 여러 도구를 호출
- **제어**: 중앙 집중식
- **용도**: 도구 기반 작업 자동화
```python
agent → [tool1, tool2, tool3, ...]
```

### Supervisor 패턴
- **구조**: 슈퍼바이저가 워커 에이전트들을 관리
- **제어**: 슈퍼바이저가 중앙에서 결정
- **용도**: 명확한 계층 구조가 필요한 경우
```python
Supervisor
    ├→ Worker Agent 1
    ├→ Worker Agent 2
    └→ Worker Agent 3
```

### Swarm 패턴
- **구조**: 에이전트 간 동적 핸드오프
- **제어**: 분산형, 각 에이전트가 다음 에이전트 결정
- **용도**: 유연한 대화 흐름, 복잡한 협업
```python
Agent A ←→ Agent B
    ↓         ↓
Agent C ←→ Agent D
```

### 비교표

| 특성 | Tool-Calling | Supervisor | Swarm |
|------|--------------|------------|-------|
| **복잡도** | 낮음 | 중간 | 중간 |
| **유연성** | 낮음 | 중간 | 높음 |
| **제어 방식** | 중앙집중 | 계층적 | 분산형 |
| **적합 사례** | 단순 작업 | 명확한 워크플로우 | 동적 대화 |
| **확장성** | 제한적 | 좋음 | 매우 좋음 |


## Swarm 핵심 API

### 주요 컴포넌트

#### 1. `create_swarm()`
멀티 에이전트 Swarm을 생성하는 메인 함수

```python
create_swarm(
    agents: list[Pregel],           # 에이전트 리스트
    default_active_agent: str,       # 기본 활성 에이전트 이름
    state_schema: StateSchemaType,   # 상태 스키마 (기본: SwarmState)
    context_schema: type[Any] | None # 컨텍스트 스키마
) -> StateGraph
```

#### 2. `SwarmState`
Swarm의 상태를 관리하는 스키마
- `MessagesState`를 상속
- 메시지 히스토리와 현재 활성 에이전트 정보 포함

#### 3. `create_handoff_tool()`
에이전트 간 제어권 전환 도구 생성

```python
create_handoff_tool(
    agent_name: str,        # 전환할 에이전트 이름
    name: str | None,       # 도구 이름 (옵션)
    description: str | None # 도구 설명 (옵션)
) -> BaseTool
```

#### 4. `add_active_agent_router()`
StateGraph에 활성 에이전트 라우터 추가

```python
add_active_agent_router(
    builder: StateGraph,
    route_to: list[str],           # 라우팅할 에이전트 이름 리스트
    default_active_agent: str      # 기본 활성 에이전트
) -> StateGraph
```


## 실습: Alice & Bob Swarm 구현



### 시나리오
- **Alice**: 수학 전문가 (덧셈 도구 보유)
- **Bob**: 해적처럼 말하는 대화 전문가
- 사용자는 대화 중 자유롭게 에이전트 전환 가능


### 필요한 패키지 설치


In [ ]:
# 필요한 패키지 설치
# !pip install -qU langgraph langgraph-swarm langchain-openai


### 환경 설정 및 라이브러리 임포트


In [ ]:
import os
from getpass import getpass

# OpenAI API 키 설정
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")


In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph_swarm import create_swarm, create_handoff_tool


### 도구 정의


In [ ]:
def add(a: int, b: int) -> int:
    """두 숫자를 더합니다."""
    return a + b

def subtract(a: int, b: int) -> int:
    """두 숫자를 뺍니다."""
    return a - b

def multiply(a: int, b: int) -> int:
    """두 숫자를 곱합니다."""
    return a * b


### 에이전트 생성

#### Alice: 수학 전문가


In [ ]:
# LLM 모델 정의
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Alice 에이전트: 수학 전문가
alice = create_react_agent(
    llm,
    tools=[
        add,
        subtract,
        multiply,
        create_handoff_tool(
            agent_name="Bob",
            description="Bob에게 대화를 전환합니다. Bob은 해적처럼 말합니다.",
        ),
    ],
    state_modifier="당신은 Alice입니다. 수학 계산 전문가이며, 친절하고 정확하게 답변합니다.",
)

print("Alice 에이전트 생성 완료")


#### Bob: 해적 대화 전문가


In [ ]:
# Bob 에이전트: 해적 스타일로 말하는 대화 전문가
bob = create_react_agent(
    llm,
    tools=[
        create_handoff_tool(
            agent_name="Alice",
            description="Alice에게 대화를 전환합니다. Alice는 수학 문제를 도와줄 수 있습니다.",
        ),
    ],
    state_modifier=(
        "당신은 Bob입니다. 항상 해적처럼 말합니다 (예: '아하르르!', '선장님!', '보물'). "
        "수학 질문을 받으면 Alice에게 전환하세요."
    ),
)

print("Bob 에이전트 생성 완료")


### Swarm 생성 및 컴파일


In [ ]:
# 메모리 체크포인터 생성
checkpointer = MemorySaver()

# Swarm 워크플로우 생성
workflow = create_swarm(
    agents=[alice, bob],
    default_active_agent="Alice",  # 기본적으로 Alice가 먼저 응답
)

# 그래프 컴파일
app = workflow.compile(checkpointer=checkpointer)

print("Swarm 애플리케이션 생성 완료")
print(f"   - 에이전트: Alice (수학), Bob (해적)")
print(f"   - 기본 에이전트: Alice")


### Swarm 그래프 시각화


In [ ]:
from IPython.display import Image, display

# 그래프 구조 시각화
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"그래프 시각화를 위해서는 graphviz 설치가 필요합니다: {e}")
    print("그래프 구조:")
    print(app.get_graph())


### 테스트 1: Alice와 수학 계산


In [ ]:
# 대화 설정
config = {"configurable": {"thread_id": "1"}}

# 첫 번째 질문: 수학 계산
print("=" * 60)
print("사용자: 5 더하기 7은 얼마야?")
print("=" * 60)

result1 = app.invoke(
    {"messages": [{"role": "user", "content": "5 더하기 7은 얼마야?"}]},
    config,
)

# 마지막 응답 출력
for msg in result1["messages"]:
    if hasattr(msg, "content") and msg.content:
        print(f"\n{msg.type}: {msg.content}")


### 테스트 2: Bob에게 전환


In [ ]:
# 두 번째 질문: Bob과 대화하기
print("\n" + "=" * 60)
print("사용자: Bob과 이야기하고 싶어")
print("=" * 60)

result2 = app.invoke(
    {"messages": [{"role": "user", "content": "Bob과 이야기하고 싶어"}]},
    config,
)

# 마지막 몇 개 메시지 출력
for msg in result2["messages"][-3:]:
    if hasattr(msg, "content") and msg.content:
        print(f"\n{msg.type}: {msg.content}")


### 테스트 3: Bob과 대화 후 다시 Alice에게


In [ ]:
# 세 번째 질문: Bob에게 수학 질문 (자동으로 Alice에게 전환)
print("\n" + "=" * 60)
print("사용자: 10 곱하기 3은?")
print("=" * 60)

result3 = app.invoke(
    {"messages": [{"role": "user", "content": "10 곱하기 3은?"}]},
    config,
)

# 마지막 몇 개 메시지 출력
for msg in result3["messages"][-4:]:
    if hasattr(msg, "content") and msg.content:
        print(f"\n{msg.type}: {msg.content}")


### 전체 대화 히스토리 확인


In [ ]:
print("\n" + "=" * 60)
print("전체 대화 히스토리")
print("=" * 60)

for i, msg in enumerate(result3["messages"], 1):
    role = msg.type if hasattr(msg, "type") else "unknown"
    content = msg.content if hasattr(msg, "content") else str(msg)
    
    # 도구 호출 메시지는 간략하게
    if "tool" in role.lower():
        print(f"\n[{i}] {role}: [도구 호출/응답]")
    else:
        print(f"\n[{i}] {role}: {content[:200]}{'...' if len(str(content)) > 200 else ''}")


## 스트리밍 모드로 실행

실시간으로 에이전트의 작업을 관찰할 수 있습니다.


In [ ]:
# 새로운 스레드로 스트리밍 테스트
config_stream = {"configurable": {"thread_id": "2"}}

print("=" * 60)
print("스트리밍 모드: 15에서 8을 빼고, 그 결과에 3을 곱해줘")
print("=" * 60)

for chunk in app.stream(
    {"messages": [{"role": "user", "content": "15에서 8을 빼고, 그 결과에 3을 곱해줘"}]},
    config_stream,
    stream_mode="values"
):
    if "messages" in chunk:
        msg = chunk["messages"][-1]
        if hasattr(msg, "content") and msg.content:
            print(f"\n[{msg.type}]")
            print(msg.content[:300])


## 핵심 개념 정리

### Swarm의 장점
1. **동적 라우팅**: 에이전트가 상황에 따라 자율적으로 다른 에이전트에게 전환
2. **컨텍스트 유지**: 대화 히스토리가 모든 에이전트에서 공유됨
3. **전문화**: 각 에이전트가 특정 작업에 최적화
4. **확장성**: 새로운 에이전트 추가가 쉬움

### 핵심 플로우
```
1. 사용자 입력 → 현재 활성 에이전트
2. 에이전트가 작업 수행 또는 핸드오프 도구 호출
3. 핸드오프 시 → 다른 에이전트로 제어권 전환
4. 새 에이전트가 작업 계속 수행
5. 응답 반환
```

### 활용 사례
- **고객 지원**: 일반 문의 → 기술 지원 → 결제 담당자
- **콘텐츠 생성**: 아이디어 → 작성 → 편집 → 검토
- **데이터 분석**: 수집 → 전처리 → 분석 → 시각화
- **코드 개발**: 설계 → 구현 → 테스트 → 배포


## 실습 과제

다음 요구사항을 만족하는 3개 에이전트 Swarm을 구현해보세요:

1. **Translator (번역가)**: 영어 ↔ 한국어 번역
2. **Writer (작가)**: 창의적인 스토리 작성
3. **Critic (평론가)**: 작성된 내용 평가 및 피드백

### 요구사항
- 각 에이전트는 적절한 핸드오프 도구를 가져야 함
- 사용자가 "번역해줘", "스토리 써줘", "평가해줘" 등의 요청으로 자연스럽게 전환
- 기본 활성 에이전트는 Writer로 설정


## 참고 자료

- [LangGraph Swarm 공식 문서](https://reference.langchain.com/python/langgraph/swarm/)
- [LangGraph 공식 문서](https://langchain-ai.github.io/langgraph/)
- [OpenAI Swarm](https://github.com/openai/swarm)

---

### 다음 학습 주제
- **고급 Swarm 패턴**: 조건부 라우팅, 병렬 에이전트
- **상태 관리**: 커스텀 SwarmState 구현
- **에러 핸들링**: 에이전트 간 예외 처리
- **성능 최적화**: 캐싱, 배치 처리
